# JAX → Jaxpr → StableHLO Practice Notebook

This notebook is a hands-on practice notebook for understanding how JAX programs are traced, represented as **Jaxpr**, lowered to **StableHLO**, and eventually compiled by XLA.

The goal is not to learn compiler theory first. The goal is to build the habit of asking:

```text
What does this JAX code become after tracing?
What does it become after lowering?
What performance implications can I infer?
```

## Learning path

```text
JAX code
  → Jaxpr
  → StableHLO / MLIR
  → XLA compilation
  → device executable
```

In this notebook, we mainly inspect:

```text
JAX code → Jaxpr → StableHLO
```

## 0. Environment setup

This notebook assumes JAX is installed. The helper functions below print:

1. Jaxpr
2. StableHLO
3. Compiled result

The key inspection API is:

```python
jax.jit(f).lower(*args).compiler_ir(dialect="stablehlo")
```

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax

print("JAX version:", jax.__version__)
print("Available devices:", jax.devices())

In [ ]:
def inspect_jax(f, *args, name=None, show_result=True):
    # Print Jaxpr, StableHLO, and optionally the result of a JAX function.
    if name:
        print(f"
{'=' * 80}
{name}
{'=' * 80}")

    print("
==== Jaxpr ====")
    try:
        print(jax.make_jaxpr(f)(*args))
    except Exception as e:
        print("[Jaxpr error]", repr(e))

    print("
==== StableHLO ====")
    try:
        lowered = jax.jit(f).lower(*args)
        print(lowered.compiler_ir(dialect="stablehlo"))
    except Exception as e:
        print("[StableHLO error]", repr(e))

    if show_result:
        print("
==== Result ====")
        try:
            print(jax.jit(f)(*args))
        except Exception as e:
            print("[Execution error]", repr(e))

## 1. Elementwise operations

Expected mapping:

```text
jnp.sin(x)  → stablehlo.sine
x + y       → stablehlo.add
y * 2       → stablehlo.multiply
```

Focus on how scalar constants such as `2.0` appear in the IR.

In [ ]:
def f_elementwise(x, y):
    return jnp.sin(x) + y * 2.0

x = jnp.ones((3,), dtype=jnp.float32)
y = jnp.arange(3, dtype=jnp.float32)

inspect_jax(f_elementwise, x, y, name="1. Elementwise: sin + multiply + add")

### Exercise 1

Modify the function and observe which StableHLO operations appear.

Try: `jnp.exp`, `jnp.log`, `jnp.tanh`, division, or chained elementwise expressions.

In [ ]:
def f_elementwise_exercise(x, y):
    # TODO: modify this expression
    return jnp.exp(x) / (y + 1.0)

inspect_jax(f_elementwise_exercise, x, y, name="Exercise 1: custom elementwise function")

## 2. Broadcasting

Broadcasting is essential when reading HLO.

```python
x.shape == (4, 3)
b.shape == (3,)
x + b
```

Look for:

```text
stablehlo.broadcast_in_dim
```

In [ ]:
def f_broadcast(x, b):
    return x + b

x2 = jnp.ones((4, 3), dtype=jnp.float32)
b2 = jnp.arange(3, dtype=jnp.float32)

inspect_jax(f_broadcast, x2, b2, name="2. Broadcasting: (4, 3) + (3,)")

### Exercise 2

Change `b2_ex` to shapes such as `(1, 3)`, `(4, 1)`, or scalar `()` and inspect the output.

In [ ]:
b2_ex = jnp.ones((1, 3), dtype=jnp.float32)
inspect_jax(f_broadcast, x2, b2_ex, name="Exercise 2: different broadcast shape")

## 3. Reshape and transpose

Many high-level tensor operations become combinations of:

```text
reshape
transpose
broadcast
slice
concatenate
```

Transformer and V-JEPA-like pipelines often contain many shape transformations around `dot_general`.

In [ ]:
def f_shape_ops(x):
    x = jnp.reshape(x, (2, 3, 4))
    x = jnp.transpose(x, (1, 0, 2))
    return jnp.reshape(x, (3, 8))

x3 = jnp.arange(24, dtype=jnp.float32)
inspect_jax(f_shape_ops, x3, name="3. Shape ops: reshape + transpose + reshape")

### Exercise 3

Add another transpose or reshape. Check whether the StableHLO gets more complex or stays simple.

In [ ]:
def f_shape_ops_exercise(x):
    # TODO: modify this chain
    x = jnp.reshape(x, (2, 3, 4))
    x = jnp.transpose(x, (2, 1, 0))
    return x

inspect_jax(f_shape_ops_exercise, x3, name="Exercise 3: custom shape ops")

## 4. Matrix multiplication and `dot_general`

JAX matrix multiplication usually lowers to:

```text
stablehlo.dot_general
```

Many operations are variations of `dot_general`: `matmul`, `einsum`, `tensordot`, batched matmul, and attention scores.

In [ ]:
def f_matmul(x, w):
    return x @ w

x4 = jnp.ones((8, 16), dtype=jnp.float32)
w4 = jnp.ones((16, 32), dtype=jnp.float32)
inspect_jax(f_matmul, x4, w4, name="4. Matmul: x @ w")

### Exercise 4: simplified attention score

Inspect:

```python
scores = q @ k.T / sqrt(d)
```

Look for `dot_general`, `transpose`, and scale operations.

In [ ]:
def f_attention_score(q, k):
    d = q.shape[-1]
    return (q @ k.T) / jnp.sqrt(jnp.asarray(d, dtype=q.dtype))

q = jnp.ones((4, 16), dtype=jnp.float32)
k = jnp.ones((6, 16), dtype=jnp.float32)
inspect_jax(f_attention_score, q, k, name="Exercise 4: simplified attention score")

## 5. Reduction operations

Expected mapping:

```text
jnp.sum(x, axis=...)   → stablehlo.reduce
jnp.mean(x, axis=...)  → stablehlo.reduce + divide
jnp.max(x, axis=...)   → stablehlo.reduce with max combiner
```

In [ ]:
def f_reduce(x):
    return jnp.mean(x, axis=1)

x5 = jnp.arange(12, dtype=jnp.float32).reshape(3, 4)
inspect_jax(f_reduce, x5, name="5. Reduction: mean over axis=1")

### Exercise 5

Try `jnp.sum`, `jnp.max`, or `jax.nn.logsumexp`. `logsumexp` is especially useful for NumPyro-style models.

In [ ]:
def f_reduce_exercise(x):
    # TODO: try sum, max, logsumexp
    return jax.nn.logsumexp(x, axis=1)

inspect_jax(f_reduce_exercise, x5, name="Exercise 5: custom reduction")

## 6. Conditional logic: `jnp.where` and `lax.cond`

Common mappings:

```text
jnp.where(cond, a, b) → stablehlo.select
lax.cond(...)         → compiler-visible control flow
```

In [ ]:
def f_where(x):
    return jnp.where(x > 0.0, x, -x)

x6 = jnp.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=jnp.float32)
inspect_jax(f_where, x6, name="6A. Elementwise conditional: jnp.where")


def f_cond(x):
    pred = jnp.sum(x) > 0.0
    return lax.cond(pred, lambda z: z * 2.0, lambda z: z - 2.0, x)

inspect_jax(f_cond, x6, name="6B. Control-flow conditional: lax.cond")

## 7. Loops: Python loop vs `lax.scan`

A Python loop inside `jit` can be unrolled when the loop count is static. `lax.scan` creates a compiler-visible loop and is usually better for long recurrent computations.

In [ ]:
def f_python_loop(x):
    y = x
    for _ in range(5):
        y = y * 1.1 + 1.0
    return y

x7 = jnp.ones((3,), dtype=jnp.float32)
inspect_jax(f_python_loop, x7, name="7A. Python loop with static range")


def f_scan(xs):
    def body(carry, x):
        carry = carry * 1.1 + x
        return carry, carry

    init = 0.0
    carry, ys = lax.scan(body, init, xs)
    return ys

xs7 = jnp.ones((5,), dtype=jnp.float32)
inspect_jax(f_scan, xs7, name="7B. lax.scan")

### Exercise 7

Increase the Python loop length and compare with `lax.scan`. Which version scales better for 100, 1,000, or 10,000 steps?

In [ ]:
def f_scan_exercise(xs):
    def body(carry, x):
        # TODO: modify recurrence
        carry = jnp.tanh(carry + x)
        return carry, carry

    carry, ys = lax.scan(body, 0.0, xs)
    return carry, ys

inspect_jax(f_scan_exercise, xs7, name="Exercise 7: custom scan")

## 8. Automatic differentiation: `grad`

`jax.grad` transforms the computation into another computation that evaluates derivatives. Inspect the additional operations generated by the backward pass.

In [ ]:
def loss_fn(w, x, y):
    pred = x @ w
    return jnp.mean((pred - y) ** 2)

grad_loss_fn = jax.grad(loss_fn)

w8 = jnp.ones((16, 1), dtype=jnp.float32)
x8 = jnp.ones((8, 16), dtype=jnp.float32)
y8 = jnp.ones((8, 1), dtype=jnp.float32)
inspect_jax(grad_loss_fn, w8, x8, y8, name="8. grad(loss_fn)")

### Exercise 8

Modify the loss: L1 loss, Huber-like loss using `jnp.where`, or logistic loss. Then inspect how the gradient changes.

In [ ]:
def custom_loss_fn(w, x, y):
    pred = x @ w
    err = pred - y
    # TODO: modify the loss
    return jnp.mean(jnp.abs(err))

custom_grad_loss_fn = jax.grad(custom_loss_fn)
inspect_jax(custom_grad_loss_fn, w8, x8, y8, name="Exercise 8: custom gradient")

## 9. Batching: `vmap`

`vmap` transforms a function so that operations themselves become batched. It is not merely a faster Python loop.

In [ ]:
def single_dot(x, w):
    return jnp.dot(x, w)

batched_dot = jax.vmap(single_dot, in_axes=(0, None))

x9 = jnp.ones((8, 16), dtype=jnp.float32)
w9 = jnp.ones((16,), dtype=jnp.float32)
inspect_jax(batched_dot, x9, w9, name="9. vmap(single_dot)")

### Exercise 9

Try `vmap(grad(f))`, which is important for per-sample gradients, influence estimation, DP-SGD, and Bayesian workflows.

In [ ]:
def loss_per_example(w, x, y):
    pred = jnp.dot(x, w)
    return (pred - y) ** 2

per_example_grad = jax.vmap(jax.grad(loss_per_example), in_axes=(None, 0, 0))

w9b = jnp.ones((16,), dtype=jnp.float32)
x9b = jnp.ones((8, 16), dtype=jnp.float32)
y9b = jnp.ones((8,), dtype=jnp.float32)
inspect_jax(per_example_grad, w9b, x9b, y9b, name="Exercise 9: vmap(grad(loss_per_example))")

## 10. Gather and scatter: GNN-style operations

GNNs frequently use indexed edge operations and aggregation:

```text
node_features[src]       → gather-like operation
out.at[dst].add(messages) → scatter-like operation
```

These operations can be performance-sensitive on GPU.

In [ ]:
def gnn_message_sum(node_features, src, dst, num_nodes):
    messages = node_features[src]
    out = jnp.zeros((num_nodes, node_features.shape[1]), dtype=node_features.dtype)
    out = out.at[dst].add(messages)
    return out

node_features = jnp.arange(5 * 3, dtype=jnp.float32).reshape(5, 3)
src = jnp.array([0, 1, 2, 3, 1, 4], dtype=jnp.int32)
dst = jnp.array([1, 2, 3, 4, 4, 0], dtype=jnp.int32)
num_nodes = 5

gnn_jitted = jax.jit(gnn_message_sum, static_argnums=(3,))

print("
==== Jaxpr ====")
print(jax.make_jaxpr(lambda nf, s, d: gnn_message_sum(nf, s, d, num_nodes))(node_features, src, dst))
print("
==== StableHLO ====")
print(gnn_jitted.lower(node_features, src, dst, num_nodes).compiler_ir(dialect="stablehlo"))
print("
==== Result ====")
print(gnn_jitted(node_features, src, dst, num_nodes))

### Exercise 10

Add edge weights and inspect whether broadcasting appears before scatter.

In [ ]:
def weighted_gnn_message_sum(node_features, src, dst, edge_weight, num_nodes):
    messages = node_features[src] * edge_weight[:, None]
    out = jnp.zeros((num_nodes, node_features.shape[1]), dtype=node_features.dtype)
    return out.at[dst].add(messages)

edge_weight = jnp.ones((src.shape[0],), dtype=jnp.float32)
weighted_gnn_jitted = jax.jit(weighted_gnn_message_sum, static_argnums=(4,))
print(weighted_gnn_jitted.lower(node_features, src, dst, edge_weight, num_nodes).compiler_ir(dialect="stablehlo"))
print(weighted_gnn_jitted(node_features, src, dst, edge_weight, num_nodes))

## 11. Static arguments and recompilation risk

JAX compilation depends on shapes, dtypes, and static arguments. If an argument controls output shape, it often needs to be static.

In [ ]:
def make_range_scaled(x, n):
    r = jnp.arange(n, dtype=x.dtype)
    return x + r

x11 = jnp.array(1.0, dtype=jnp.float32)

try:
    print(jax.jit(make_range_scaled)(x11, 4))
except Exception as e:
    print("Expected issue when n is not static:")
    print(repr(e))

make_range_scaled_jit = jax.jit(make_range_scaled, static_argnums=(1,))
print(make_range_scaled_jit(x11, 4))
print(make_range_scaled_jit.lower(x11, 4).compiler_ir(dialect="stablehlo"))

for n in [4, 8, 16]:
    print("n =", n, "result =", make_range_scaled_jit(x11, n))

## 12. Mini case study: Bayesian-style log probability

A lot of NumPyro-like computation is tensor algebra plus reduction:

```text
linear algebra
broadcasting
log / exp
reduction
```

In [ ]:
def gaussian_log_likelihood(beta, x, y, sigma):
    mu = x @ beta
    resid = y - mu
    logp = -0.5 * (resid / sigma) ** 2 - jnp.log(sigma) - 0.5 * jnp.log(2.0 * jnp.pi)
    return jnp.sum(logp)

beta12 = jnp.ones((4,), dtype=jnp.float32)
x12 = jnp.ones((10, 4), dtype=jnp.float32)
y12 = jnp.ones((10,), dtype=jnp.float32)
sigma12 = jnp.array(1.5, dtype=jnp.float32)
inspect_jax(gaussian_log_likelihood, beta12, x12, y12, sigma12, name="12. Gaussian log likelihood")

### Exercise 12

Extend the model by adding intercept, per-observation sigma, prior log probability, or negative log posterior.

In [ ]:
def gaussian_log_posterior_exercise(beta, intercept, x, y, sigma):
    mu = x @ beta + intercept
    resid = y - mu
    log_lik = -0.5 * (resid / sigma) ** 2 - jnp.log(sigma) - 0.5 * jnp.log(2.0 * jnp.pi)
    log_prior_beta = -0.5 * jnp.sum(beta ** 2)
    return jnp.sum(log_lik) + log_prior_beta

intercept12 = jnp.array(0.1, dtype=jnp.float32)
inspect_jax(
    gaussian_log_posterior_exercise,
    beta12,
    intercept12,
    x12,
    y12,
    sigma12,
    name="Exercise 12: Gaussian log posterior",
)

## 13. Reading checklist

When you inspect StableHLO, use this checklist:

```text
1. Are the expected high-level operations present?
   - dot_general
   - reduce
   - gather
   - scatter
   - while

2. Are there many shape transformations?
   - reshape
   - transpose
   - broadcast_in_dim

3. Is a Python loop unrolled?
   - many repeated operations may indicate unrolling

4. Are shape-controlling values static?
   - static_argnums may be needed

5. Are gather/scatter operations central?
   - common in GNNs, sparse workflows, indexing-heavy code

6. Could changing input shapes trigger recompilation?
   - JAX compiles per shape/dtype/static-arg signature
```

## 14. Suggested next practice

After finishing this notebook, apply the same inspection workflow to your own code:

```text
1. A small GNN message passing function
2. A NumPyro log_prob-like function
3. A Transformer attention block
4. A time-series recurrence with lax.scan
5. A vmap + grad function
```

The target skill is:

```text
Given JAX code, predict the rough Jaxpr and StableHLO structure before printing it.
```

That is the practical bridge from writing JAX to understanding how XLA sees your program.